# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets with @id and name
print("Available Record Sets:")
record_sets = dataset.list_record_sets()
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")

# Choose a record set for detailed field overview
record_set_id = record_sets[0]['@id'] if record_sets else None
if record_set_id:
    print(f"\nFields for record set {record_set_id}:")
    fields = dataset.list_fields(record_set=record_set_id)
    for f in fields:
        print(f"- @id: {f['@id']}, name: {f.get('name', '(no name)')}, dataType: {f.get('dataType', '(unknown)')}")
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from the record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all data from all record sets
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Record set @id: {rs_id}, loaded dataframe with shape: {df.shape}")

# Display columns of the first record set
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    print(f"\nColumns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, standardizing numeric fields, and grouping data.

In [ ]:
# Example: select a numeric field (use its @id from the previous overview)
# You may need to adapt these IDs and fields depending on what's present in your dataset.

# Get list of numeric fields (@id and type)
fields = dataset.list_fields(record_set=main_record_set_id)
numeric_fields = [f for f in fields if f.get('dataType', '').lower() in ('number', 'integer', 'float')]
if not numeric_fields:
    print("No numeric fields found.")
else:
    numeric_field_id = numeric_fields[0]['@id']
    numeric_field_name = numeric_fields[0]['name']
    df = dataframes[main_record_set_id]
    print(f"Using numeric field: @id={numeric_field_id}, name={numeric_field_name}")
    
    # Remove missing or invalid values (if any)
    df_clean = df.copy()
    df_clean = df_clean[pd.to_numeric(df_clean[numeric_field_id], errors='coerce').notnull()]
    df_clean[numeric_field_id] = pd.to_numeric(df_clean[numeric_field_id], errors='coerce')

    # Example threshold for filtering
    threshold = df_clean[numeric_field_id].mean()

    filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[numeric_field_id + '_normalized'] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() > 0 else 1)
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Try grouping by another categorical field
    # Find the first non-numeric, non-NA field for grouping
    group_field = None
    for f in fields:
        if f['@id'] != numeric_field_id and df[f['@id']].nunique() < len(df) // 2:
            group_field = f['@id']
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_fields:
    print("No numeric field for plotting.")
else:
    # Distribution plot of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df_clean[numeric_field_id], kde=True, bins=16, color='skyblue')
    plt.title(f'Distribution of {numeric_field_name} (@id={numeric_field_id})')
    plt.xlabel(numeric_field_name)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group field if present
    if group_field:
        plt.figure(figsize=(9, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df_clean)
        plt.title(f'{numeric_field_name} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_name)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to load, explore, and perform basic analysis on the FAIR² dataset using the `mlcroissant` library. We enumerated the available record sets and fields (referenced by their `@id` values), extracted tabular data, conducted simple EDA steps such as filtering and normalization, and visualized numeric fields. For further analysis, more domain knowledge of each field and record set can be leveraged, and the workflow can be extended with domain-specific processing and advanced visualizations.